# Vérification empirique — rapprochement `commandes_clients` ↔ `commandes` (FluxPro)

**Compétence couverte : C13 — Modéliser un entrepôt de données**
**Épreuve associée : E5**

Ce notebook formalise, avec preuve chiffrée, une décision déjà actée sans
mesure formelle dans
[`modelisation_merise.md` §4.2](../docs/architecture/modelisation_merise.md#42-commandes_clients-reste-indépendante-de-commandes-fluxpro) :
`commandes_clients` (C10/C11, commandes brutes des clients) et `commandes`
(FluxPro, schéma fourni) ne peuvent pas être rapprochées de manière fiable
avec les données disponibles dans ce programme.

Deux vérifications indépendantes :
1. La clé candidate `(client, entrepot, date_commande)` est ambiguë côté
   FluxPro — inutilisable comme clé de jointure.
2. `omega_historique_expeditions.csv` ne porte aucun identifiant capable de
   servir de pont entre les deux jeux de commandes (pas d'id de commande,
   pas de SKU, pas de quantité).

**Conclusion anticipée** (vérifiée ci-dessous) : `Dim_Commande` (bloc 3)
devra traiter les commandes clients et les commandes FluxPro comme deux
faits distincts, non joignables au niveau ligne.


## 1. Ambiguïté de la clé `(client, entrepot, date_commande)` côté FluxPro

Si l'on tentait de rapprocher `commandes_clients` (une commande par
`(client, commande_id)`, sans lien connu vers `commandes.id`) et
`commandes` (FluxPro) via une clé composite plausible — client, entrepôt,
date — encore faudrait-il que cette clé identifie une commande FluxPro de
façon unique. Vérification sur les 1400 commandes FluxPro réelles.


In [1]:
import csv
from collections import Counter
from pathlib import Path

RAW = Path("..") / "data" / "raw"

with open(RAW / "clients.csv") as f:
    clients_by_id = {r["id"]: r["nom"] for r in csv.DictReader(f)}

with open(RAW / "entrepots.csv") as f:
    entrepots_by_id = {r["id"]: r["ville"] for r in csv.DictReader(f)}

with open(RAW / "commandes.csv") as f:
    commandes = list(csv.DictReader(f))

print(f"Commandes FluxPro chargées : {len(commandes)}")
print("Colonnes :", list(commandes[0].keys()))


Commandes FluxPro chargées : 1400
Colonnes : ['id', 'client_id', 'entrepot_id', 'date_commande', 'statut']


In [2]:
# Clé candidate en libellés lisibles (client, ville d'entrepôt, date),
# directement comparable au format texte libre utilisé par l'historique.
keys = [
    (clients_by_id[c["client_id"]], entrepots_by_id[c["entrepot_id"]], c["date_commande"])
    for c in commandes
]
key_counts = Counter(keys)

n_distinct = len(key_counts)
ambiguous = {k: n for k, n in key_counts.items() if n > 1}
n_ambiguous = len(ambiguous)
rows_in_ambiguous_keys = sum(ambiguous.values())

print(f"Clés (client, entrepot, date_commande) distinctes : {n_distinct}")
print(f"Clés partagées par plusieurs commandes le même jour : {n_ambiguous}")
print(f"Taux de clés ambiguës : {100 * n_ambiguous / n_distinct:.1f} %")
print(f"Commandes concernées par une clé ambiguë : {rows_in_ambiguous_keys} / {len(commandes)}"
      f" ({100 * rows_in_ambiguous_keys / len(commandes):.1f} %)")


Clés (client, entrepot, date_commande) distinctes : 1223
Clés partagées par plusieurs commandes le même jour : 166
Taux de clés ambiguës : 13.6 %
Commandes concernées par une clé ambiguë : 343 / 1400 (24.5 %)


In [3]:
# Exemples concrets de clés ambiguës, pour illustrer le risque de jointure
print("Exemples de clés partagées par plusieurs commandes distinctes :")
for k, n in list(ambiguous.items())[:5]:
    client, entrepot, date = k
    print(f"  ({client}, {entrepot}, {date}) -> {n} commandes FluxPro différentes")


Exemples de clés partagées par plusieurs commandes distinctes :
  (NordDrive, Marseille, 2025-12-23) -> 2 commandes FluxPro différentes
  (NordDrive, Lille, 2025-12-04) -> 2 commandes FluxPro différentes
  (MedioTex, Marseille, 2026-04-19) -> 2 commandes FluxPro différentes
  (NordDrive, Lyon, 2025-01-25) -> 2 commandes FluxPro différentes
  (MedioTex, Marseille, 2026-03-24) -> 2 commandes FluxPro différentes


**Lecture** : 13,6 % des clés `(client, entrepot, date_commande)`
distinctes correspondent en réalité à *plusieurs* commandes FluxPro le même
jour (jusqu'à 24,5 % des commandes prises individuellement sont concernées
par une clé non unique). Une jointure sur cette clé rattacherait donc, dans
un cas sur huit environ, une commande cliente à la mauvaise commande
FluxPro (ou échouerait à choisir laquelle) — inutilisable comme clé
d'appariement fiable, ce qui confirme empiriquement la réserve déjà émise
dans `modelisation_merise.md` §4.2 ("jointure arbitraire, trop fragile").


## 2. `omega_historique_expeditions.csv` ne peut pas servir de pont

Autre piste envisageable : utiliser l'historique des expéditions comme
table de passage entre `commandes_clients` et `commandes` (FluxPro),
puisqu'il porte lui aussi un champ `client`. Vérification de son schéma
réel.


In [4]:
with open(RAW / "historique" / "omega_historique_expeditions.csv") as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    historique = list(reader)

print(f"Lignes de l'historique : {len(historique)}")
print("Colonnes disponibles :", columns)

champs_necessaires_pour_pont = {"commande_id", "sku", "quantite"}
presents = champs_necessaires_pour_pont & set(columns)
print(f"Champs nécessaires à un pont commande/ligne présents : {presents or 'aucun'}")


Lignes de l'historique : 25000
Colonnes disponibles : ['id', 'client', 'entrepot', 'categorie_produit', 'date_expedition', 'poids_kg', 'delai_livraison_jours', 'cout_transport_eur', 'statut']
Champs nécessaires à un pont commande/ligne présents : aucun


In [5]:
print("Exemple de ligne réelle :")
print(historique[0])


Exemple de ligne réelle :
{'id': '1', 'client': 'MedioTex', 'entrepot': 'Marseille', 'categorie_produit': 'Textile', 'date_expedition': '2024-04-11', 'poids_kg': '29.83', 'delai_livraison_jours': '1', 'cout_transport_eur': '52.37', 'statut': 'Retardee'}


**Lecture** : l'historique ne porte ni identifiant de commande, ni SKU,
ni quantité — seulement `client` (texte libre), `entrepot` (texte libre,
ville), `categorie_produit` (catégorie, pas un produit précis),
`date_expedition` (date d'expédition, pas la date de commande), et des
mesures agrégées (`poids_kg`, `delai_livraison_jours`, `cout_transport_eur`,
`statut`). Structurellement, il ne peut relier une commande client à une
ligne de commande FluxPro : il n'y a rien à joindre au niveau ligne, et le
seul champ commun envisageable (`client`) souffre en plus de la même
limite de texte libre déjà documentée en C9/C11
(`modelisation_merise.md` §4.3).


## 3. Conclusion

Les deux vérifications convergent : ni une clé composite `(client,
entrepot, date)` (ambiguë sur FluxPro), ni l'historique des expéditions
(dépourvu de tout identifiant de commande/ligne) ne permettent de
rapprocher fiablement `commandes_clients` et `commandes` (FluxPro) au
niveau ligne.

**Décision confirmée pour C13** : `Dim_Commande` (et les faits associés,
`Fait_Commande` côté clients et `Fait_Expedition`/`Fait_Stock` côté
FluxPro) traiteront les commandes clients et les commandes FluxPro comme
**deux faits distincts**, sans tenter de rapprochement arbitraire au
niveau ligne — cohérent avec la modélisation déjà retenue en C11
(`modelisation_merise.md` §4.2), désormais appuyée par une preuve
chiffrée plutôt que par une seule affirmation.
